# PATSTAT — yearly F/E/G + Disruption trend

The twin of `PatentView/notebook/patent_feg_disruption_trend.ipynb`: mean **CD**, **F/E/G** and the type-j
fraction by **filing year**, per window, from `patstat_disruption.parquet` (applications with a defined
`CD_all`, i.e. at least one citer).

## Output
`PATSTAT/output/patstat_feg_disruption_trend.parquet` — one row per filing year: `year, n, CD_w_mean, F_w_mean,
E_w_mean, G_w_mean, ni_w_mean, nj_w_mean, nk_w_mean, njfrac_w_mean` for `w in (_3,_5,_10,_all)`.

In [ ]:
import os, sys, gc, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PATSTAT')
import ps_common as ps
OUT = ps.OUT
OUT_FP = ps.out('patstat_feg_disruption_trend.parquet')
ps.preflight('patstat_feg_disruption_trend')

import matplotlib.pyplot as plt
%matplotlib inline
DISR, META = ps.out('patstat_disruption.parquet'), ps.out('patstat_metadata.parquet')
WIN = ['_3', '_5', '_10', '_all']; MET = ['CD', 'F', 'E', 'G', 'ni', 'nj', 'nk']
con = ps.connect()

## 1. Aggregate by filing year

In [ ]:
%%time
sel = []
for w in WIN:
    sel += [f'avg({m}{w}) AS {m}{w}_mean' for m in MET]
    sel.append(f'avg(nj{w} / nullif(ni{w} + nj{w} + nk{w}, 0)) AS njfrac{w}_mean')
tr = con.execute(f"""SELECT m.filing_year AS year, count(*) AS n, {', '.join(sel)}
  FROM read_parquet('{DISR}') d JOIN read_parquet('{META}') m USING (appln_id)
  WHERE d.CD_all IS NOT NULL AND m.filing_year BETWEEN 1900 AND {ps.SNAP_YEAR} GROUP BY 1 ORDER BY 1""").fetchdf()
tr.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(tr):,} years, {len(tr.columns)} cols)')
display(tr.tail(10).round(4))
con.close()

## 2. Plot

In [ ]:
d = tr[(tr['year'] >= 1950) & (tr['year'] <= ps.SNAP_YEAR - 5)]
fig, ax = plt.subplots(1, 4, figsize=(20, 4.3))
for w, col in zip(WIN, ['#55a868', '#c44e52', '#dd8452', '#4c72b0']):
    ax[0].plot(d['year'], d[f'CD{w}_mean'], label=f'CD{w}', color=col)
ax[0].axhline(0, color='k', lw=0.6); ax[0].set_title('Mean disruption (CD) by filing year'); ax[0].set_ylabel('mean CD'); ax[0].legend()
for m, col in zip(['F', 'E', 'G'], ['#4c72b0', '#dd8452', '#55a868']):
    ax[1].plot(d['year'], d[f'{m}_all_mean'], label=m, color=col)
ax[1].set_title('Mean F/E/G (all) by filing year'); ax[1].set_ylabel('fraction'); ax[1].legend()
ax[2].stackplot(d['year'], d['F_all_mean'], d['E_all_mean'], d['G_all_mean'], labels=['F', 'E', 'G'], colors=['#4c72b0', '#dd8452', '#55a868'], alpha=0.85)
ax[2].set_title('F/E/G composition (all) by filing year'); ax[2].set_ylim(0, 1); ax[2].legend(loc='upper left')
for nm, col in zip(['ni', 'nj', 'nk'], ['#4c72b0', '#dd8452', '#55a868']):
    ax[3].plot(d['year'], d[f'{nm}_all_mean'], label=nm, color=col)
ax[3].set_yscale('log'); ax[3].set_title('Mean ni/nj/nk (all) by filing year'); ax[3].set_ylabel('mean count (log)'); ax[3].legend()
plt.tight_layout(); plt.show()